# Hybrid CNN + Transformer VQA Notebook

> **Project idea:** build a stronger blind-assistance VQA/classification pipeline by combining a pretrained **ConvNeXtSmall CNN** with lightweight **Transformer attention blocks**.  
> The CNN extracts strong visual features, the Transformer refines image tokens and question tokens, and cross-attention fuses both modalities before answer classification.

**Why this notebook uses ConvNeXt + Transformer**

- Uses a stronger pretrained CNN backbone: **ConvNeXtSmall**.
- Replaces GRU text modeling with a **Transformer Encoder**.
- Adds **visual Transformer blocks** over CNN feature-map tokens.
- Adds **image-question cross-attention** before classification.
- Includes comparison, evaluation, error analysis, reproducibility notes, and a GUI/TTS demo.


## Rubric Roadmap

| Rubric criterion | How this notebook covers it |
|---|---|
| Problem understanding & task framing | We frame the task as visual question answering / image-label prediction for blind-assistance images with special priority for traffic signs and signals. |
| Model selection & architecture justification | ConvNeXtSmall handles strong visual feature extraction; Transformers model attention over visual tokens and question tokens; cross-attention fuses both modalities. |
| Implementation quality | Clean data pipeline, answer filtering, class merging, class-balanced modeling set, stratified train/validation/test split, augmentation, class weights, callbacks, checkpointing, and saved mappings. |
| Use of deep learning concepts | CNN transfer learning, embeddings, positional encodings, self-attention, cross-attention, fine-tuning, regularization, top-k accuracy. |
| Experimentation & comparative analysis | Frozen-backbone vs fine-tuned-backbone comparison, top-1/top-5 metrics, balanced accuracy, macro F1, confusion matrix, per-class accuracy, error table. |
| Innovation & intellectual contribution | Hybrid CNN-Transformer design specialized for the scraped sign + VizWiz setting, with image-only deployment support and imbalance-aware class design. |
| Interpretation & error analysis | Failure inspection, class-weight visibility, and limitations section after testing. |
| Reproducibility & notebook quality | Fixed seed, centralized configuration, saved artifacts, clear section order, and Markdown explanations. |


## 0. Environment Setup

Run this notebook from top to bottom. In Colab, use **Runtime -> Change runtime type -> GPU** for faster training.


In [ ]:
!pip install -q requests datasets pillow tensorflow scikit-learn matplotlib pandas ipywidgets deep-translator gtts


In [ ]:
import os
import random
import re
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import ImageEnhance
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display
from tensorflow.keras import layers, Model
from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess_input

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

if tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled for GPU training.")


## 1. Data Collection: Scraped Traffic Signs + VizWiz

The project needs realistic blind-assistance images and road-safety examples. We keep the original scraping pipeline, then merge scraped sign/signal samples with a 2,000-image VizWiz validation subset.


In [ ]:
from pathlib import Path

# Sign/signal targets for the blind-assistance project.
# Open Images has reliable box labels for these road-safety classes.
SCRAPE_JOBS = [
    {
        "label": "stop sign",
        "openimages_label_ids": ["/m/02pv19"],
        "openimages_classes": ["Stop sign"],
    },
    {
        "label": "traffic light",
        "openimages_label_ids": ["/m/015qff"],
        "openimages_classes": ["Traffic light"],
    },
    {
        "label": "traffic sign",
        "openimages_label_ids": ["/m/01mqdt"],
        "openimages_classes": ["Traffic sign"],
    },
]

# Keep this True for the final project so training cannot silently skip scraping.
REQUIRE_SCRAPED_DATA = True

# Save scraped outputs beside these notebooks.
NOTEBOOK_DIR = Path.cwd()  # Run VS Code/Jupyter from the folder that contains these notebooks.
SCRAPED_IMAGE_DIR = str(NOTEBOOK_DIR / "scraped_sign_vqa_images")
SCRAPED_METADATA_PATH = str(NOTEBOOK_DIR / "scraped_sign_vqa_metadata.jsonl")
SCRAPED_DATASET_PATH = str(NOTEBOOK_DIR / "scraped_sign_vqa_dataset")
OPEN_IMAGES_CACHE_DIR = str(NOTEBOOK_DIR / "openimages_cache")

TARGET_SCRAPED_IMAGES = 450
MAX_IMAGES_PER_CATEGORY = 150
MAX_IMAGES_PER_URL = MAX_IMAGES_PER_CATEGORY  # Less than 2000 scraped images by design.


In [ ]:
import csv
import hashlib
import json
import time
from io import BytesIO
from pathlib import Path

import requests
from datasets import Dataset, Image as HFImage
from PIL import Image as PILImage, UnidentifiedImageError
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


OPEN_IMAGES_SPLITS = [
    {
        "name": "validation",
        "annotations_url": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
        "images_url": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",
    },
    {
        "name": "test",
        "annotations_url": "https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv",
        "images_url": "https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv",
    },
]

SCRAPER_VERSION = "openimages-signs-direct-s3-v2"
COMMONS_USER_AGENT = "BlindAssistSignScraper/1.0 (educational image dataset builder)"
REQUEST_TIMEOUT = (5, 60)  # Metadata CSV downloads can be large.
IMAGE_REQUEST_TIMEOUT = (3, 8)  # Individual thumbnails should fail fast.
MAX_IMAGE_DOWNLOAD_BYTES = 3 * 1024 * 1024
SAVED_IMAGE_MAX_SIDE = 768
MIN_IMAGE_SIDE = 64

PILImage.MAX_IMAGE_PIXELS = None


def _make_requests_session():
    retry = Retry(
        total=2,
        connect=2,
        read=2,
        status=2,
        backoff_factor=0.75,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=16, pool_maxsize=16)
    session = requests.Session()
    session.headers.update({"User-Agent": COMMONS_USER_AGENT})
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def _download_file(session, url, cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / url.rsplit("/", 1)[-1]

    if path.exists() and path.stat().st_size > 0:
        print(f"Using cached metadata: {path}", flush=True)
        return path

    print(f"Downloading metadata: {url}", flush=True)
    try:
        with session.get(url, timeout=REQUEST_TIMEOUT, stream=True) as response:
            response.raise_for_status()
            total = int(response.headers.get("Content-Length") or 0)
            downloaded = 0
            next_report = 25 * 1024 * 1024
            with path.open("wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)
                    if downloaded >= next_report:
                        if total:
                            print(f"  metadata {downloaded / 1024 / 1024:.0f}/{total / 1024 / 1024:.0f} MB", flush=True)
                        else:
                            print(f"  metadata {downloaded / 1024 / 1024:.0f} MB", flush=True)
                        next_report += 25 * 1024 * 1024
    except requests.exceptions.RequestException as exc:
        if path.exists():
            path.unlink()
        raise RuntimeError(f"Could not download Open Images metadata from {url}: {exc}") from exc

    return path


def _job_label(job):
    return str(job.get("label", "scraped image")).strip().lower()


def _openimages_class_map(scrape_jobs):
    class_to_labels = {}
    labels = []
    for job in scrape_jobs:
        label = _job_label(job)
        labels.append(label)
        for class_id in job.get("openimages_label_ids", []):
            class_to_labels.setdefault(class_id, []).append(label)
    return labels, class_to_labels


def _collect_openimages_candidates(session, scrape_jobs, cache_dir):
    labels, class_to_labels = _openimages_class_map(scrape_jobs)
    candidates = {label: [] for label in labels}
    seen_for_label = {label: set() for label in labels}
    needed_image_ids_by_split = {split["name"]: set() for split in OPEN_IMAGES_SPLITS}

    for split in OPEN_IMAGES_SPLITS:
        annotation_path = _download_file(session, split["annotations_url"], cache_dir)
        print(f"Scanning Open Images annotations: {split['name']}", flush=True)
        with annotation_path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                matched_labels = class_to_labels.get(row.get("LabelName"))
                if not matched_labels:
                    continue

                image_id = row.get("ImageID")
                if not image_id:
                    continue

                for label in matched_labels:
                    if image_id in seen_for_label[label]:
                        continue
                    seen_for_label[label].add(image_id)
                    needed_image_ids_by_split[split["name"]].add(image_id)
                    candidates[label].append({
                        "image_id": image_id,
                        "split": split["name"],
                        "openimages_label_id": row.get("LabelName"),
                    })

    for label in labels:
        print(f"Open Images candidates for {label}: {len(candidates[label])}", flush=True)

    return candidates, needed_image_ids_by_split


def _load_openimages_urls(session, needed_image_ids_by_split, cache_dir):
    # Direct Open Images files are much more reliable than old Flickr thumbnail URLs.
    # Example: https://open-images-dataset.s3.amazonaws.com/validation/<ImageID>.jpg
    image_urls = {}
    base_url = "https://open-images-dataset.s3.amazonaws.com"

    for split in OPEN_IMAGES_SPLITS:
        split_name = split["name"]
        needed_ids = needed_image_ids_by_split.get(split_name, set())
        if not needed_ids:
            continue

        print(f"Generating direct Open Images S3 URLs for {split_name}: {len(needed_ids)} images", flush=True)
        for image_id in needed_ids:
            image_urls[(split_name, image_id)] = {
                "url": f"{base_url}/{split_name}/{image_id}.jpg",
                "license": "",
                "landing_url": "",
                "title": image_id,
                "author": "Open Images Dataset",
            }

    return image_urls




def _download_and_save_image(session, image_url, output_dir, min_side=MIN_IMAGE_SIDE):
    chunks = []
    downloaded = 0

    try:
        with session.get(image_url, timeout=IMAGE_REQUEST_TIMEOUT, stream=True) as response:
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "").lower()
            if "image" not in content_type:
                return None

            content_length = int(response.headers.get("Content-Length") or 0)
            if content_length and content_length > MAX_IMAGE_DOWNLOAD_BYTES:
                return None

            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue
                downloaded += len(chunk)
                if downloaded > MAX_IMAGE_DOWNLOAD_BYTES:
                    return None
                chunks.append(chunk)
    except requests.exceptions.RequestException:
        return None

    if not chunks:
        return None

    try:
        with PILImage.open(BytesIO(b"".join(chunks))) as image:
            image = image.convert("RGB")
            if min(image.size) < int(min_side):
                return None

            image.thumbnail((SAVED_IMAGE_MAX_SIDE, SAVED_IMAGE_MAX_SIDE))
            digest = hashlib.md5(image_url.encode("utf-8")).hexdigest()
            image_path = output_dir / f"{digest}.jpg"
            image.save(image_path, format="JPEG", quality=92, optimize=True)
            return {"image_path": str(image_path)}
    except (UnidentifiedImageError, OSError, ValueError):
        return None


def run_openimages_scrape(
    scrape_jobs=SCRAPE_JOBS,
    output_dir=SCRAPED_IMAGE_DIR,
    output_jsonl=SCRAPED_METADATA_PATH,
    target_images=TARGET_SCRAPED_IMAGES,
    max_images_per_category=MAX_IMAGES_PER_CATEGORY,
    cache_dir=OPEN_IMAGES_CACHE_DIR,
    require_scraped_data=REQUIRE_SCRAPED_DATA,
):
    output_dir = Path(output_dir)
    output_jsonl = Path(output_jsonl)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_jsonl.write_text("", encoding="utf-8")

    if not scrape_jobs:
        message = "SCRAPE_JOBS is empty. Add sign/signal Open Images labels before training."
        if require_scraped_data:
            raise ValueError(message)
        print(message)
        return str(output_jsonl)

    session = _make_requests_session()
    labels, _class_to_labels = _openimages_class_map(scrape_jobs)

    print(f"Using Open Images sign/signal scraper. Version: {SCRAPER_VERSION}", flush=True)
    print("First run downloads metadata CSV files; later runs reuse /content/openimages_cache.", flush=True)
    print("Collecting only sign/signal labels configured in SCRAPE_JOBS.", flush=True)

    try:
        candidates, needed_image_ids_by_split = _collect_openimages_candidates(session, scrape_jobs, cache_dir)
        image_urls = _load_openimages_urls(session, needed_image_ids_by_split, cache_dir)
    except RuntimeError as exc:
        if require_scraped_data:
            raise
        print(exc)
        return str(output_jsonl)

    total_saved = 0
    counts_by_label = {label: 0 for label in labels}
    attempts_by_label = {label: 0 for label in labels}
    positions = {label: 0 for label in labels}
    seen_image_ids = set()
    seen_image_urls = set()

    def save_next_for_label(label, ignore_label_limit=False):
        nonlocal total_saved

        if total_saved >= int(target_images):
            return False
        if not ignore_label_limit and counts_by_label[label] >= int(max_images_per_category):
            return False

        while positions[label] < len(candidates[label]):
            candidate = candidates[label][positions[label]]
            positions[label] += 1

            image_key = (candidate["split"], candidate["image_id"])
            if image_key in seen_image_ids:
                continue

            url_info = image_urls.get(image_key)
            if not url_info:
                continue

            image_url = url_info["url"]
            if image_url in seen_image_urls:
                continue

            attempts_by_label[label] += 1
            if attempts_by_label[label] <= 5 or attempts_by_label[label] % 10 == 0:
                print(
                    f"  trying {label}: attempt {attempts_by_label[label]}, "
                    f"saved {counts_by_label[label]}, total {total_saved}",
                    flush=True,
                )

            download_result = _download_and_save_image(session, image_url, output_dir)
            if download_result is None:
                if attempts_by_label[label] <= 5 or attempts_by_label[label] % 25 == 0:
                    print(f"  skipped slow/bad image for {label}: attempt {attempts_by_label[label]}", flush=True)
                continue

            image_path = download_result["image_path"]
            seen_image_ids.add(image_key)
            seen_image_urls.add(image_url)

            record = {
                "image_path": image_path,
                "question": "what sign or traffic signal is shown",
                "answer": label,
                "all_answers": [label],
                "source_dataset": "OpenImagesSigns",
                "image_id": candidate["image_id"],
                "split": candidate["split"],
                "openimages_label_id": candidate["openimages_label_id"],
                "page_url": url_info.get("landing_url", ""),
                "image_url": image_url,
                "license": url_info.get("license", ""),
                "title": url_info.get("title", ""),
                "author": url_info.get("author", ""),
            }
            with output_jsonl.open("a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

            total_saved += 1
            counts_by_label[label] += 1
            if counts_by_label[label] <= 5 or counts_by_label[label] % 10 == 0 or total_saved % 50 == 0:
                print(
                    f"  saved {label}: {counts_by_label[label]} images, total: {total_saved}",
                    flush=True,
                )

            time.sleep(0.02)
            return True

        return False

    print("Balanced pass: up to", max_images_per_category, "images per label", flush=True)
    for label in labels:
        print(f"Collecting label: {label} ({len(candidates[label])} candidates)", flush=True)
        while total_saved < int(target_images) and counts_by_label[label] < int(max_images_per_category):
            if not save_next_for_label(label, ignore_label_limit=False):
                break
        print(f"  label images used: {counts_by_label[label]}, total scraped rows: {total_saved}", flush=True)

    if total_saved < int(target_images):
        print(f"Fill pass: adding extra images from labels with more candidates until total reaches {target_images}.", flush=True)
        made_progress = True
        while total_saved < int(target_images) and made_progress:
            made_progress = False
            for label in labels:
                if total_saved >= int(target_images):
                    break
                if save_next_for_label(label, ignore_label_limit=True):
                    made_progress = True

    if total_saved == 0:
        message = (
            "No scraped records found. Open Images metadata was reachable, but no image files "
            "could be downloaded. Check your internet connection and retry."
        )
        if require_scraped_data:
            raise RuntimeError(message)
        print(message)

    print("Open Images sign/signal scraped metadata saved to:", output_jsonl)
    print("Total Open Images sign/signal scraped rows:", total_saved)
    print("Rows by label:", counts_by_label)
    return str(output_jsonl)


# Backward-compatible wrappers for older notebook calls.
def run_wikimedia_commons_api_scrape(*args, **kwargs):
    print("Wikimedia Commons is skipped on this runtime; using Open Images instead.", flush=True)
    return run_openimages_scrape(*args, **kwargs)


def run_wikimedia_commons_scrape(*args, **kwargs):
    print("Wikimedia Commons is skipped on this runtime; using Open Images instead.", flush=True)
    return run_openimages_scrape(*args, **kwargs)


def run_scrapy_image_scrape(
    scrape_jobs=SCRAPE_JOBS,
    output_dir=SCRAPED_IMAGE_DIR,
    output_jsonl=SCRAPED_METADATA_PATH,
    max_images_per_url=MAX_IMAGES_PER_URL,
    require_scraped_data=REQUIRE_SCRAPED_DATA,
):
    return run_openimages_scrape(
        scrape_jobs=scrape_jobs,
        output_dir=output_dir,
        output_jsonl=output_jsonl,
        max_images_per_category=max_images_per_url,
        require_scraped_data=require_scraped_data,
    )


def load_scraped_dataset(metadata_path=SCRAPED_METADATA_PATH, require_scraped_data=REQUIRE_SCRAPED_DATA):
    metadata_path = Path(metadata_path)
    if not metadata_path.exists() or metadata_path.stat().st_size == 0:
        message = "No scraped sign/signal records found. The Open Images scraper did not collect images."
        if require_scraped_data:
            raise RuntimeError(message)
        print(message)
        return None

    rows = []
    for line in metadata_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if Path(row["image_path"]).exists() and str(row.get("answer", "")).strip():
            answer = str(row.get("answer", "")).strip().lower()
            rows.append({
                "image": row["image_path"],
                "question": row.get("question", "what is in the image"),
                "answer": answer,
                "all_answers": row.get("all_answers") or [answer],
                "source_dataset": row.get("source_dataset", "OpenImagesSigns"),
            })

    if not rows:
        message = "Scraped metadata exists, but no valid image rows were found."
        if require_scraped_data:
            raise RuntimeError(message)
        print(message)
        return None

    scraped_dataset = Dataset.from_dict({
        "image": [row["image"] for row in rows],
        "question": [row["question"] for row in rows],
        "answer": [row["answer"] for row in rows],
        "all_answers": [row["all_answers"] for row in rows],
        "source_dataset": [row["source_dataset"] for row in rows],
    }).cast_column("image", HFImage())

    scraped_dataset.save_to_disk(SCRAPED_DATASET_PATH)
    print("Scraped dataset rows:", len(scraped_dataset))
    print("Saved scraped dataset to:", SCRAPED_DATASET_PATH)
    return scraped_dataset


run_openimages_scrape()
scraped_dataset = load_scraped_dataset()
print("Final scraped rows:", 0 if scraped_dataset is None else len(scraped_dataset))


## 2. Merge With VizWiz 2,000 Sample

VizWiz contributes real user-taken accessibility images. The scraped sign data makes sure the traffic-sign classes are represented and not lost among common VizWiz answers.


In [ ]:
from datasets import load_dataset, concatenate_datasets
from collections import Counter

N_VIZWIZ = 2000

vizwiz_train = load_dataset(
    "lmms-lab/VizWiz-VQA",
    split=f"val[:{N_VIZWIZ}]"
)

print(vizwiz_train)
print("VizWiz columns:", vizwiz_train.column_names)
print("VizWiz sample size:", len(vizwiz_train))


In [ ]:
from datasets import concatenate_datasets
from collections import Counter

def extract_answer_text(ans):
    if ans is None:
        return ""

    if isinstance(ans, dict):
        return str(ans.get("answer", "")).strip().lower()

    return str(ans).strip().lower()


def clean_answers_list(answers):
    if answers is None:
        return []

    clean = []
    for ans in answers:
        txt = extract_answer_text(ans)
        if txt != "":
            clean.append(txt)

    return clean


def get_majority_answer(answers):
    clean = clean_answers_list(answers)

    if len(clean) == 0:
        return ""

    return Counter(clean).most_common(1)[0][0]


In [ ]:
def add_vizwiz_cols(answers):
    clean_answers = clean_answers_list(answers)
    return {
        "answer": get_majority_answer(answers),
        "all_answers": clean_answers,
        "source_dataset": "VizWiz"
    }


In [ ]:
vizwiz_clean = vizwiz_train.map(
    add_vizwiz_cols,
    input_columns=["answers"]
)


In [ ]:
keep_cols = ["image", "question", "answer", "all_answers", "source_dataset"]

vizwiz_clean = vizwiz_clean.select_columns(keep_cols)


In [ ]:
datasets_to_merge = [vizwiz_clean]

if "scraped_dataset" in globals() and scraped_dataset is not None and len(scraped_dataset) > 0:
    datasets_to_merge.append(scraped_dataset)
    print("Merging scraped rows:", len(scraped_dataset))
else:
    print("No scraped rows to merge. Using VizWiz sample only.")

combined_dataset = concatenate_datasets(datasets_to_merge)
combined_dataset = combined_dataset.shuffle(seed=42)

print(combined_dataset)
print("Rows after VizWiz + scraped merge:", len(combined_dataset))
combined_dataset


## 3. Data Inspection

We inspect the merged dataset first. The final train/validation/test split is rebuilt later **after** answer cleaning and class balancing, so every reported class has real support in each split.


In [ ]:
sample = combined_dataset[0]

print("Source:", sample["source_dataset"])
print("Question:", sample["question"])
print("Answer:", sample["answer"])
print("All answers:", sample["all_answers"])


In [ ]:
save_path = "/content/hybrid_vqa_dataset"
combined_dataset.save_to_disk(save_path)
print("Saved merged dataset to:", save_path)


In [ ]:
sample = combined_dataset[0]

sample["image"]


In [ ]:
print(combined_dataset)

print("Columns:", combined_dataset.column_names)
print("Number of rows:", len(combined_dataset))


In [ ]:
for i in range(3):
    sample = combined_dataset[i]
    print("----- Sample", i, "-----")
    print("Source:", sample["source_dataset"])
    print("Question:", sample["question"])
    print("Answer:", sample["answer"])
    print()


In [ ]:
combined_dataset = combined_dataset.filter(
    lambda x: x["answer"] is not None and str(x["answer"]).strip() != ""
)

print(combined_dataset)
print("Rows after filtering:", len(combined_dataset))


In [ ]:
# Quick exploratory split only.
# The final modeling split is rebuilt after class merging and balancing in Section 4.
split_1 = combined_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_1["train"]
temp_dataset = split_1["test"]

split_2 = temp_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

val_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Exploratory Train:", train_dataset)
print("Exploratory Validation:", val_dataset)
print("Exploratory Test:", test_dataset)
print("Final stratified modeling split is created after class design.")


In [ ]:
print("Train rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))
print("Test rows:", len(test_dataset))

print("\nSample from train:")
print(train_dataset[0]["question"])
print(train_dataset[0]["answer"])

print("\nSample from validation:")
print(val_dataset[0]["question"])
print(val_dataset[0]["answer"])

print("\nSample from test:")
print(test_dataset[0]["question"])
print(test_dataset[0]["answer"])


In [ ]:
print("Train rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))
print("Test rows:", len(test_dataset))


## 4. Answer Cleaning, Class Merging, and Balanced Class Design

This section fixes the imbalance problem at the data-design level.

We do four things before model training:

1. Remove weak/empty labels.
2. Merge small semantic labels such as individual colors into broader classes.
3. Move unsupported rare labels into a controlled `other object` bucket.
4. Build a class-balanced modeling dataset and a stratified split so the report never contains classes with `support = 0`.


In [ ]:
bad_answers = {
    "",
    "unanswerable",
    "unsuitable image",
    "can't tell",
    "cannot tell",
    "not sure",
    "unknown",
    "none",
    "nothing",
}

MERGED_ANSWER_GROUPS = {
    "color": {
        "grey",
        "gray",
        "blue",
        "brown",
        "black",
        "white",
        "red",
        "green",
        "yellow",
        "orange",
        "purple",
        "pink",
    },
    "digital screen": {
        "computer screen",
        "computer",
        "laptop",
        "keyboard",
        "phone",
        "cell phone",
        "television",
        "installing windows",
        "please select boot device",
    },
    "food or drink": {
        "water",
        "beans",
        "bottle",
        "food package",
        "medicine bottle",
    },
    "text or number": {
        "1 877 328 9677",
        "number",
        "text",
    },
}


def normalize_answer_text(answer):
    answer = str(answer).strip().lower()
    answer = re.sub(r"\s+", " ", answer)
    return answer


def canonical_answer_text(answer):
    answer = normalize_answer_text(answer)
    for merged_label, variants in MERGED_ANSWER_GROUPS.items():
        if answer in variants:
            return merged_label
    return answer


def remove_bad_answers(example):
    ans = normalize_answer_text(example["answer"])
    return ans not in bad_answers and len(ans) > 0


def merge_small_answer_classes(example):
    merged_answer = canonical_answer_text(example["answer"])
    raw_all_answers = example.get("all_answers", [])
    if raw_all_answers is None:
        raw_all_answers = []
    if isinstance(raw_all_answers, str):
        raw_all_answers = [raw_all_answers]

    merged_all_answers = [
        canonical_answer_text(answer)
        for answer in raw_all_answers
        if canonical_answer_text(answer) not in bad_answers
    ]
    if not merged_all_answers:
        merged_all_answers = [merged_answer]

    return {
        "answer": merged_answer,
        "all_answers": merged_all_answers,
    }


def label_distribution_table(dataset, title, top_n=30):
    labels = [normalize_answer_text(item["answer"]) for item in dataset]
    counts = Counter(labels)
    total = sum(counts.values())
    rows = [
        {
            "answer": answer,
            "count": count,
            "percent": round(100 * count / total, 2) if total else 0.0,
        }
        for answer, count in counts.most_common()
    ]
    df = pd.DataFrame(rows)
    print(f"{title}: {len(counts)} classes, {total} samples")
    if counts:
        min_count = min(counts.values())
        max_count = max(counts.values())
        imbalance_ratio = max_count / max(1, min_count)
        print(f"Min count: {min_count} | Max count: {max_count} | Imbalance ratio: {imbalance_ratio:.2f}x")
    display(df.head(top_n))
    return df


combined_filtered_dataset = combined_dataset.filter(remove_bad_answers, load_from_cache_file=False)
print("Rows after bad-answer filtering:", len(combined_filtered_dataset))
distribution_before_merge = label_distribution_table(combined_filtered_dataset, "Full label distribution before merging")

combined_clean_dataset = combined_filtered_dataset.map(merge_small_answer_classes, load_from_cache_file=False)
print("Merged answer groups:", MERGED_ANSWER_GROUPS)
distribution_after_merge = label_distribution_table(combined_clean_dataset, "Full label distribution after semantic merging")


In [ ]:
TOP_K = 14
MIN_CLASS_TOTAL_COUNT = 8
MAX_SAMPLES_PER_CLASS = 160
RARE_CLASS_LABEL = "other object"
ENABLE_RARE_CLASS_BUCKET = True
SIGN_SOURCE_DATASETS = {"OpenImagesSigns", "OpenImages", "Scraped"}
SIGN_PRIORITY_ANSWERS = {
    "stop sign",
    "traffic light",
    "traffic signal",
    "traffic sign",
    "street sign",
    "sign",
    "stop",
}


def dataset_label_counts(dataset):
    return Counter(normalize_answer_text(item["answer"]) for item in dataset)


answer_counter = dataset_label_counts(combined_clean_dataset)
frequent_answers = [
    answer
    for answer, count in answer_counter.most_common()
    if count >= MIN_CLASS_TOTAL_COUNT
]

sign_answers = sorted({
    normalize_answer_text(item["answer"])
    for item in combined_clean_dataset
    if (
        item.get("source_dataset", "") in SIGN_SOURCE_DATASETS
        or normalize_answer_text(item["answer"]) in SIGN_PRIORITY_ANSWERS
    )
    and answer_counter[normalize_answer_text(item["answer"])] >= 3
})

selected_answers = []
for answer in sign_answers:
    if answer and answer not in selected_answers:
        selected_answers.append(answer)

for answer in frequent_answers:
    if answer not in selected_answers:
        selected_answers.append(answer)
    if len(selected_answers) >= TOP_K:
        break


def apply_model_class_policy(example):
    answer = normalize_answer_text(example["answer"])
    if answer not in selected_answers and ENABLE_RARE_CLASS_BUCKET:
        answer = RARE_CLASS_LABEL
    return {
        "answer": answer,
        "all_answers": [answer],
    }


model_candidate_dataset = combined_clean_dataset.map(apply_model_class_policy, load_from_cache_file=False)

model_counts = dataset_label_counts(model_candidate_dataset)
final_answers = [
    answer
    for answer, count in model_counts.most_common()
    if count >= MIN_CLASS_TOTAL_COUNT
]

for answer in selected_answers:
    if answer in model_counts and answer not in final_answers:
        final_answers.append(answer)

if ENABLE_RARE_CLASS_BUCKET and RARE_CLASS_LABEL in model_counts and RARE_CLASS_LABEL not in final_answers:
    final_answers.append(RARE_CLASS_LABEL)


def valid_for_model(example):
    return normalize_answer_text(example["answer"]) in set(final_answers)


model_dataset = model_candidate_dataset.filter(valid_for_model, load_from_cache_file=False)


def cap_samples_per_class(dataset, max_samples_per_class=160, seed=42):
    rng = np.random.default_rng(seed)
    by_label = {}
    for source_index, item in enumerate(dataset):
        label = normalize_answer_text(item["answer"])
        by_label.setdefault(label, []).append(source_index)

    selected_indices = []
    for label, label_indices in by_label.items():
        label_indices = np.array(label_indices, dtype=np.int32)
        if len(label_indices) > max_samples_per_class:
            label_indices = rng.choice(label_indices, size=max_samples_per_class, replace=False)
        selected_indices.extend(label_indices.tolist())

    rng.shuffle(selected_indices)
    return dataset.select(selected_indices)


model_dataset = cap_samples_per_class(model_dataset, max_samples_per_class=MAX_SAMPLES_PER_CLASS, seed=SEED)
model_distribution = label_distribution_table(model_dataset, "Final balanced modeling label distribution")

final_answers = [answer for answer in model_distribution["answer"].tolist()]
answer2idx = {answer: idx for idx, answer in enumerate(final_answers)}
idx2answer = {idx: answer for answer, idx in answer2idx.items()}


def stratified_model_split(dataset, train_ratio=0.80, val_ratio=0.10, seed=42):
    rng = np.random.default_rng(seed)
    by_label = {}
    for source_index, item in enumerate(dataset):
        label = normalize_answer_text(item["answer"])
        by_label.setdefault(label, []).append(source_index)

    train_indices, val_indices, test_indices = [], [], []
    split_rows = []

    for label, label_indices in sorted(by_label.items()):
        label_indices = np.array(label_indices, dtype=np.int32)
        rng.shuffle(label_indices)
        n = len(label_indices)
        if n < 3:
            continue

        test_count = max(1, int(round(n * (1.0 - train_ratio - val_ratio))))
        val_count = max(1, int(round(n * val_ratio)))
        if test_count + val_count >= n:
            test_count = 1
            val_count = 1

        test_part = label_indices[:test_count]
        val_part = label_indices[test_count:test_count + val_count]
        train_part = label_indices[test_count + val_count:]

        train_indices.extend(train_part.tolist())
        val_indices.extend(val_part.tolist())
        test_indices.extend(test_part.tolist())

        split_rows.append({
            "answer": label,
            "total": n,
            "train": len(train_part),
            "validation": len(val_part),
            "test": len(test_part),
        })

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)

    split_table = pd.DataFrame(split_rows).sort_values("total", ascending=False)
    return (
        dataset.select(train_indices),
        dataset.select(val_indices),
        dataset.select(test_indices),
        split_table,
    )


train_cls, val_cls, test_cls, stratified_split_table = stratified_model_split(model_dataset, seed=SEED)
train_dataset, val_dataset, test_dataset = train_cls, val_cls, test_cls

print("Number of final answer classes:", len(answer2idx))
print("Final answer classes:", list(answer2idx.keys()))
print("Train rows for model:", len(train_cls))
print("Validation rows for model:", len(val_cls))
print("Test rows for model:", len(test_cls))
display(stratified_split_table)

zero_support_classes = stratified_split_table.query("train == 0 or validation == 0 or test == 0")
if len(zero_support_classes) == 0:
    print("Balanced split check passed: every reported class has train/validation/test support.")
else:
    display(zero_support_classes)


## 5. Question Encoding for the Transformer

The model supports real VQA questions and also supports an image-only deployment question for the final GUI. If the goal is pure image classification, set `USE_FIXED_QUESTION_FOR_TRAINING = True`. For stronger rubric coverage, the default uses the original question text.


In [ ]:
USE_FIXED_QUESTION_FOR_TRAINING = False
FIXED_QUESTION = "what is in the image"
DEPLOYMENT_QUESTION = "what sign or traffic signal is shown"
MAX_QUESTION_LEN = 24
MAX_VOCAB_SIZE = 4000


def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()


def get_model_question(item):
    if USE_FIXED_QUESTION_FOR_TRAINING:
        return FIXED_QUESTION
    question = str(item.get("question", "")).strip()
    return question if question else FIXED_QUESTION


word_counter = Counter()
for item in train_cls:
    word_counter.update(tokenize(get_model_question(item)))
word_counter.update(tokenize(FIXED_QUESTION))
word_counter.update(tokenize(DEPLOYMENT_QUESTION))

word2idx = {"<pad>": 0, "<unk>": 1}
for word, _count in word_counter.most_common(MAX_VOCAB_SIZE - 2):
    word2idx[word] = len(word2idx)


def encode_question(question=FIXED_QUESTION):
    tokens = tokenize(question)
    ids = [word2idx.get(token, word2idx["<unk>"]) for token in tokens[:MAX_QUESTION_LEN]]
    ids += [word2idx["<pad>"]] * (MAX_QUESTION_LEN - len(ids))
    return np.array(ids, dtype=np.int32)


num_answers = len(answer2idx)
vocab_size = len(word2idx)

print("Number of answer classes:", num_answers)
print("Vocab size:", vocab_size)
print("Training question mode:", "fixed image-only question" if USE_FIXED_QUESTION_FOR_TRAINING else "original VQA questions")
print("Deployment question:", DEPLOYMENT_QUESTION)


## 6. Image Pipeline, Oversampling, and Class Weights

The final modeling set is already class-balanced and stratified. Training still uses two extra protections:

1. **Oversampling:** minority classes are duplicated in the training-only view and augmented differently each epoch.
2. **Class weights:** `compute_class_weight` gives rare labels a larger loss contribution during training.


In [ ]:
MODEL_IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8
USE_OVERSAMPLING = True
USE_CLASS_WEIGHT = True
OVERSAMPLE_TARGET_PERCENTILE = 90
MAX_CLASS_WEIGHT = 4.0


def augment_pil_image(image):
    if random.random() < 0.50:
        angle = random.uniform(-6, 6)
        image = image.rotate(angle, resample=2, fillcolor=(255, 255, 255))
    if random.random() < 0.50:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.20))
    if random.random() < 0.50:
        image = ImageEnhance.Brightness(image).enhance(random.uniform(0.90, 1.12))
    return image


class OversampledDatasetView:
    # Duplicates minority-class indices without copying image data.

    def __init__(self, hf_dataset, answer2idx, target_count, seed=42):
        self.dataset = hf_dataset
        self.answer2idx = answer2idx
        self.target_count = int(target_count)
        rng = np.random.default_rng(seed)

        by_label = {}
        for source_index, item in enumerate(hf_dataset):
            label = answer2idx[normalize_answer_text(item["answer"])]
            by_label.setdefault(label, []).append(source_index)

        indices = []
        for label, label_indices in by_label.items():
            indices.extend(label_indices)
            missing = max(0, self.target_count - len(label_indices))
            if missing > 0:
                indices.extend(rng.choice(label_indices, size=missing, replace=True).tolist())

        self.indices = np.array(indices, dtype=np.int32)
        rng.shuffle(self.indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return self.dataset[int(self.indices[int(index)])]


class HybridVQADataGenerator(tf.keras.utils.Sequence):
    def __init__(self, hf_dataset, answer2idx, batch_size=8, shuffle=True, augment=False, **kwargs):
        super().__init__(**kwargs)
        self.dataset = hf_dataset
        self.answer2idx = answer2idx
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.dataset))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataset) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        images, questions, labels = [], [], []

        for dataset_index in batch_indices:
            item = self.dataset[int(dataset_index)]
            image = item["image"].convert("RGB").resize(MODEL_IMAGE_SIZE)
            if self.augment:
                image = augment_pil_image(image)

            image_array = np.array(image, dtype=np.float32)
            image_array = convnext_preprocess_input(image_array)
            answer = normalize_answer_text(item["answer"])
            label = self.answer2idx[answer]

            images.append(image_array)
            questions.append(encode_question(get_model_question(item)))
            labels.append(label)

        inputs = {
            "image_input": np.array(images, dtype=np.float32),
            "question_input": np.array(questions, dtype=np.int32),
        }
        labels = np.array(labels, dtype=np.int32)
        return inputs, labels


train_model_distribution = label_distribution_table(train_cls, "Train distribution before oversampling")

train_labels_for_weights = np.array(
    [answer2idx[normalize_answer_text(item["answer"])] for item in train_cls],
    dtype=np.int32,
)
classes_for_weights = np.array(sorted(np.unique(train_labels_for_weights)), dtype=np.int32)
computed_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_for_weights,
    y=train_labels_for_weights,
)
class_weight_dict = {
    int(class_idx): float(min(weight, MAX_CLASS_WEIGHT))
    for class_idx, weight in zip(classes_for_weights, computed_class_weights)
}
class_weight_df = pd.DataFrame([
    {
        "answer": idx2answer[int(class_idx)],
        "class_index": int(class_idx),
        "class_weight": round(float(class_weight_dict[int(class_idx)]), 4),
    }
    for class_idx in classes_for_weights
]).sort_values("class_weight", ascending=False)

print("Highest class weights")
display(class_weight_df.head(15))

label_counts = Counter(train_labels_for_weights.tolist())
target_count = int(np.percentile(list(label_counts.values()), OVERSAMPLE_TARGET_PERCENTILE))
target_count = max(target_count, min(max(label_counts.values()), 4))

if USE_OVERSAMPLING:
    train_balanced = OversampledDatasetView(train_cls, answer2idx, target_count=target_count, seed=SEED)
else:
    train_balanced = train_cls

print("Original train samples:", len(train_cls))
print("Training samples after oversampling view:", len(train_balanced))
print("Oversampling target count per minority class:", target_count)
train_oversampled_distribution = label_distribution_table(train_balanced, "Train distribution after oversampling")

train_gen = HybridVQADataGenerator(train_balanced, answer2idx, batch_size=BATCH_SIZE, shuffle=True, augment=True)
val_gen = HybridVQADataGenerator(val_cls, answer2idx, batch_size=BATCH_SIZE, shuffle=False, augment=False)
test_gen = HybridVQADataGenerator(test_cls, answer2idx, batch_size=BATCH_SIZE, shuffle=False, augment=False)

batch_x, batch_y = train_gen[0]
print(batch_x["image_input"].shape)
print(batch_x["question_input"].shape)
print(batch_y.shape)
print("Class weights enabled:", USE_CLASS_WEIGHT)


## 7. Hybrid Model: ConvNeXtSmall + Visual/Text Transformers

**Architecture justification**

1. **ConvNeXtSmall CNN** learns strong visual representations from ImageNet transfer learning.
2. The final CNN feature map is reshaped into visual tokens, then refined by **Transformer Encoder** blocks.
3. The question is embedded with learned position embeddings and processed by another **Transformer Encoder**.
4. **Cross-attention** lets visual tokens attend to question tokens before classification.
5. Training is staged: first freeze the CNN, then fine-tune only the CNN tail with a small learning rate.


In [ ]:
@tf.keras.utils.register_keras_serializable(package="HybridVQA")
class AddLearnedPositionEmbedding(layers.Layer):
    def __init__(self, max_length, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_length = max_length
        self.embed_dim = embed_dim
        self.position_embedding = layers.Embedding(input_dim=max_length, output_dim=embed_dim)
        self.supports_masking = True

    def call(self, inputs):
        length = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=length, delta=1)
        return inputs + self.position_embedding(positions)

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({"max_length": self.max_length, "embed_dim": self.embed_dim})
        return config


@tf.keras.utils.register_keras_serializable(package="HybridVQA")
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout)
        self.ffn_dense1 = layers.Dense(ff_dim, activation="gelu")
        self.ffn_dropout = layers.Dropout(dropout)
        self.ffn_dense2 = layers.Dense(embed_dim)
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
        self.supports_masking = True

    def call(self, inputs, training=False, mask=None):
        attention_mask = None
        if mask is not None:
            attention_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.bool)
        attn_output = self.att(inputs, inputs, attention_mask=attention_mask, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn_dense1(out1)
        ffn_output = self.ffn_dropout(ffn_output, training=training)
        ffn_output = self.ffn_dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout": self.dropout_rate,
        })
        return config


In [ ]:
CNN_BACKBONE = "ConvNeXtSmall"
TRANSFORMER_DIM = 256
NUM_HEADS = 4
VISUAL_TRANSFORMER_BLOCKS = 2
TEXT_TRANSFORMER_BLOCKS = 2
FF_DIM = 512
DROPOUT = 0.25
MAX_VISUAL_TOKENS = 64


def make_cnn_backbone(name, image_input):
    common_kwargs = dict(include_top=False, weights="imagenet", include_preprocessing=True, input_tensor=image_input)
    if name == "ConvNeXtSmall":
        return tf.keras.applications.ConvNeXtSmall(**common_kwargs)
    if name == "ConvNeXtTiny":
        return tf.keras.applications.ConvNeXtTiny(**common_kwargs)
    raise ValueError(f"Unsupported CNN backbone: {name}")


image_input = layers.Input(shape=(MODEL_IMAGE_SIZE[0], MODEL_IMAGE_SIZE[1], 3), name="image_input")
base_cnn = make_cnn_backbone(CNN_BACKBONE, image_input)
base_cnn.trainable = False

feature_map = base_cnn.output
visual_tokens = layers.Reshape((-1, int(feature_map.shape[-1])), name="cnn_feature_tokens")(feature_map)
visual_tokens = layers.Dense(TRANSFORMER_DIM, name="visual_token_projection")(visual_tokens)
visual_tokens = AddLearnedPositionEmbedding(MAX_VISUAL_TOKENS, TRANSFORMER_DIM, name="visual_position_embedding")(visual_tokens)
for block_idx in range(VISUAL_TRANSFORMER_BLOCKS):
    visual_tokens = TransformerEncoderBlock(TRANSFORMER_DIM, NUM_HEADS, FF_DIM, DROPOUT, name=f"visual_transformer_{block_idx + 1}")(visual_tokens)
visual_features = layers.GlobalAveragePooling1D(name="visual_token_pooling")(visual_tokens)

question_input = layers.Input(shape=(MAX_QUESTION_LEN,), name="question_input")
question_tokens = layers.Embedding(vocab_size, TRANSFORMER_DIM, mask_zero=True, name="question_embedding")(question_input)
question_tokens = AddLearnedPositionEmbedding(MAX_QUESTION_LEN, TRANSFORMER_DIM, name="question_position_embedding")(question_tokens)
for block_idx in range(TEXT_TRANSFORMER_BLOCKS):
    question_tokens = TransformerEncoderBlock(TRANSFORMER_DIM, NUM_HEADS, FF_DIM, DROPOUT, name=f"question_transformer_{block_idx + 1}")(question_tokens)
question_features = layers.GlobalAveragePooling1D(name="question_token_pooling")(question_tokens)

cross_tokens = layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=TRANSFORMER_DIM // NUM_HEADS,
    dropout=DROPOUT,
    name="image_question_cross_attention",
)(query=visual_tokens, value=question_tokens, key=question_tokens)
cross_tokens = layers.Add(name="cross_attention_residual")([visual_tokens, cross_tokens])
cross_tokens = layers.LayerNormalization(epsilon=1e-6, name="cross_attention_norm")(cross_tokens)
cross_features = layers.GlobalAveragePooling1D(name="cross_attention_pooling")(cross_tokens)

combined = layers.Concatenate(name="hybrid_feature_fusion")([visual_features, question_features, cross_features])
x = layers.Dense(512, activation="gelu")(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.45)(x)
x = layers.Dense(256, activation="gelu")(x)
x = layers.Dropout(0.30)(x)
output = layers.Dense(num_answers, activation="softmax", dtype="float32", name="answer_classifier")(x)

model = Model(inputs={"image_input": image_input, "question_input": question_input}, outputs=output, name="ConvNeXtSmall_Transformer_VQA")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy")],
)

model.summary()


## 8. Training Strategy

Stage 1 trains the new Transformer/fusion/classifier layers while the CNN is frozen. Stage 2 fine-tunes only the last CNN layers with a much smaller learning rate. The notebook keeps the stage that gives the best validation accuracy.


In [ ]:
MODEL_ARTIFACT_PATH = "/content/convnext_transformer_vqa_model.keras"
MAPPING_ARTIFACT_PATH = "/content/convnext_transformer_vqa_mappings.pkl"
STAGE2_CANDIDATE_PATH = "/content/convnext_transformer_stage2_candidate.keras"

STAGE1_EPOCHS = 12
STAGE2_EPOCHS = 8
STAGE2_MIN_IMPROVEMENT = 0.002
FINE_TUNE_LAST_N_LAYERS = 35


def compile_model_for_training(learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy")],
    )


def make_training_callbacks(checkpoint_path, early_patience, lr_patience):
    return [
        tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=early_patience, min_delta=STAGE2_MIN_IMPROVEMENT, mode="max", restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=lr_patience, min_lr=1e-6, verbose=1),
    ]


compile_model_for_training(learning_rate=3e-4)
stage1_callbacks = make_training_callbacks(MODEL_ARTIFACT_PATH, early_patience=3, lr_patience=2)

fit_kwargs = {"class_weight": class_weight_dict} if USE_CLASS_WEIGHT else {}

history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    **fit_kwargs,
)
stage1_best_val = max(history_stage1.history.get("val_accuracy", [0.0]))
print(f"Best Stage 1 val_accuracy: {stage1_best_val:.4f}")

base_cnn.trainable = True
for layer in base_cnn.layers[:-FINE_TUNE_LAST_N_LAYERS]:
    layer.trainable = False
for layer in base_cnn.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model_for_training(learning_rate=5e-6)
stage2_callbacks = make_training_callbacks(STAGE2_CANDIDATE_PATH, early_patience=2, lr_patience=1)

history_stage2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=STAGE2_EPOCHS,
    callbacks=stage2_callbacks,
    **fit_kwargs,
)
stage2_best_val = max(history_stage2.history.get("val_accuracy", [0.0]))
print(f"Best Stage 2 val_accuracy: {stage2_best_val:.4f}")

if stage2_best_val > stage1_best_val + STAGE2_MIN_IMPROVEMENT and os.path.exists(STAGE2_CANDIDATE_PATH):
    model = tf.keras.models.load_model(STAGE2_CANDIDATE_PATH, compile=False)
    model.save(MODEL_ARTIFACT_PATH)
    selected_stage = "stage2 fine-tuned ConvNeXtSmall model"
else:
    model = tf.keras.models.load_model(MODEL_ARTIFACT_PATH, compile=False)
    selected_stage = "stage1 frozen ConvNeXtSmall model"

compile_model_for_training(learning_rate=5e-6)
print("Selected best model:", selected_stage)
print("Final model saved to:", MODEL_ARTIFACT_PATH)

history = {
    "stage1": history_stage1.history,
    "stage2": history_stage2.history,
    "stage1_best_val_accuracy": stage1_best_val,
    "stage2_best_val_accuracy": stage2_best_val,
    "selected_stage": selected_stage,
}


In [ ]:
with open(MAPPING_ARTIFACT_PATH, "wb") as f:
    pickle.dump({
        "word2idx": word2idx,
        "idx2answer": idx2answer,
        "answer2idx": answer2idx,
        "MAX_QUESTION_LEN": MAX_QUESTION_LEN,
        "FIXED_QUESTION": FIXED_QUESTION,
        "DEPLOYMENT_QUESTION": DEPLOYMENT_QUESTION,
        "USE_FIXED_QUESTION_FOR_TRAINING": USE_FIXED_QUESTION_FOR_TRAINING,
        "MODEL_IMAGE_SIZE": MODEL_IMAGE_SIZE,
        "CNN_BACKBONE": CNN_BACKBONE,
        "architecture": "ConvNeXtSmall + visual Transformer + text Transformer + cross-attention",
        "MERGED_ANSWER_GROUPS": MERGED_ANSWER_GROUPS,
        "MIN_CLASS_TOTAL_COUNT": MIN_CLASS_TOTAL_COUNT,
        "MAX_SAMPLES_PER_CLASS": MAX_SAMPLES_PER_CLASS,
        "RARE_CLASS_LABEL": RARE_CLASS_LABEL,
        "USE_OVERSAMPLING": USE_OVERSAMPLING,
        "USE_CLASS_WEIGHT": USE_CLASS_WEIGHT,
        "class_weight_dict": class_weight_dict,
    }, f)

print("Saved model artifact:", MODEL_ARTIFACT_PATH)
print("Saved mapping artifact:", MAPPING_ARTIFACT_PATH)


## 9. Evaluation and Comparative Analysis

We evaluate top-1 accuracy, top-5 accuracy, balanced accuracy, macro F1, confusion patterns, and per-class behavior.

Because the split is stratified after class design, the classification report below includes only real test-supported classes and avoids misleading `support = 0` rows.


In [ ]:
test_loss, test_acc, test_top5 = model.evaluate(test_gen, verbose=1)

hybrid_transformer_result = {
    "model": "ConvNeXtSmall + visual/text Transformers + cross-attention",
    "answer_classes": len(answer2idx),
    "selected_stage": selected_stage,
    "test_accuracy": float(test_acc),
    "test_top5_accuracy": float(test_top5),
    "test_loss": float(test_loss),
}

hybrid_transformer_result


In [ ]:
stage_comparison = pd.DataFrame([
    {"experiment": "Stage 1: frozen CNN, train Transformer/fusion head", "best_val_accuracy": stage1_best_val},
    {"experiment": "Stage 2: fine-tune last CNN layers", "best_val_accuracy": stage2_best_val},
])
stage_comparison


In [ ]:
def plot_history_metric(history_dict, metric):
    plt.figure(figsize=(8, 4))
    for stage_name, stage_history in [("stage1", history_dict["stage1"]), ("stage2", history_dict["stage2"])]:
        if metric in stage_history:
            plt.plot(stage_history[metric], label=f"{stage_name}_{metric}")
        val_metric = f"val_{metric}"
        if val_metric in stage_history:
            plt.plot(stage_history[val_metric], label=f"{stage_name}_{val_metric}")
    plt.title(metric)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


plot_history_metric(history, "accuracy")
plot_history_metric(history, "loss")


In [ ]:
def collect_predictions(generator):
    y_true, y_pred, y_conf = [], [], []
    for batch_idx in range(len(generator)):
        inputs, labels = generator[batch_idx]
        preds = model.predict(inputs, verbose=0)
        y_true.extend(labels.tolist())
        y_pred.extend(np.argmax(preds, axis=1).tolist())
        y_conf.extend(np.max(preds, axis=1).tolist())
    return np.array(y_true), np.array(y_pred), np.array(y_conf)


y_true, y_pred, y_conf = collect_predictions(test_gen)
labels_present = sorted(np.unique(y_true).tolist())
target_names = [idx2answer[i] for i in labels_present]

balanced_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=labels_present, average="weighted", zero_division=0)

hybrid_transformer_result.update({
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
})

print(f"Raw accuracy: {test_acc:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels_present,
    target_names=target_names,
    zero_division=0,
    output_dict=True,
)
classification_report_df = pd.DataFrame(report_dict).transpose()
display(classification_report_df.round(4))

predicted_outside_test = sorted(set(y_pred.tolist()) - set(y_true.tolist()))
assert len(predicted_outside_test) == 0, "Every final class should have test support because the split is stratified."


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels_present)
plt.figure(figsize=(max(8, len(labels_present) * 0.45), max(6, len(labels_present) * 0.35)))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(labels_present)), target_names, rotation=90)
plt.yticks(range(len(labels_present)), target_names)
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
per_class_rows = []
for label_idx in labels_present:
    mask = y_true == label_idx
    correct = int(np.sum(y_pred[mask] == label_idx))
    total = int(np.sum(mask))
    per_class_rows.append({
        "answer": idx2answer[label_idx],
        "test_count": total,
        "class_accuracy": correct / total if total else 0.0,
        "class_weight": round(float(class_weight_dict.get(int(label_idx), 1.0)), 4),
    })


per_class_accuracy_df = pd.DataFrame(per_class_rows).sort_values(
    ["class_accuracy", "test_count"],
    ascending=[True, False],
)
per_class_accuracy_df.head(15)


## 10. Error Analysis and Imbalance Reflection

The table below shows misclassified test examples. The extra columns show whether the failed class was rare and how much class weighting tried to protect it.


In [ ]:
test_label_counts = Counter(y_true.tolist())
error_rows = []
for row_idx, (true_idx, pred_idx, conf) in enumerate(zip(y_true, y_pred, y_conf)):
    if int(true_idx) == int(pred_idx):
        continue
    item = test_cls[int(row_idx)]
    error_rows.append({
        "row": row_idx,
        "question": item.get("question", ""),
        "source_dataset": item.get("source_dataset", ""),
        "true_answer": idx2answer[int(true_idx)],
        "predicted_answer": idx2answer[int(pred_idx)],
        "confidence": round(float(conf), 4),
        "true_test_count": int(test_label_counts[int(true_idx)]),
        "true_class_weight": round(float(class_weight_dict.get(int(true_idx), 1.0)), 4),
    })


if error_rows:
    display(pd.DataFrame(error_rows).sort_values(["true_test_count", "confidence"], ascending=[True, False]).head(15))
else:
    print("No misclassified examples in the current test split.")


### Reflection

- **Strengths:** ConvNeXtSmall gives strong pretrained visual features, while Transformer attention helps fuse visual tokens with question tokens.
- **Imbalance handling:** tiny labels are semantically merged, unsupported rare labels are folded into `other object`, majority classes are capped for the modeling dataset, train/validation/test are stratified, minority training examples are oversampled with augmentation, and `compute_class_weight` increases the loss contribution for remaining smaller classes.
- **Why accuracy is not enough:** if one label dominates the dataset, raw accuracy can hide poor rare-class performance. Balanced accuracy, macro F1, per-class accuracy, and the error table are therefore reported.
- **Expected failure modes:** ambiguous VizWiz images, dark/blurry uploads, rare visual concepts inside `other object`, and labels that require OCR or external knowledge.
- **Next improvements:** add OCR features, collect more real examples for weak classes, compare ConvNeXtTiny/Small/Base, and add Grad-CAM or attention visualization for interpretability.


## 11. Inference Helpers and GUI/TTS Demo

The GUI lets you upload an image, optionally edit the hidden question, and generate English/Arabic audio. This keeps the final product aligned with the blind-assistance use case.


In [ ]:
import base64
import io
import tempfile

from deep_translator import GoogleTranslator
from gtts import gTTS
from IPython.display import HTML, clear_output, display
from PIL import Image, ImageFilter, ImageOps

MODEL_NAME = "ConvNeXtSmall + Transformer high-accuracy"
SCRAPY_MATCH_THRESHOLD = 0.90

if ("idx2answer" not in globals() or "word2idx" not in globals()) and os.path.exists(MAPPING_ARTIFACT_PATH):
    with open(MAPPING_ARTIFACT_PATH, "rb") as f:
        mappings = pickle.load(f)
    answer2idx = mappings["answer2idx"]
    idx2answer = mappings["idx2answer"]
    word2idx = mappings["word2idx"]
    MAX_QUESTION_LEN = mappings["MAX_QUESTION_LEN"]
    FIXED_QUESTION = mappings.get("FIXED_QUESTION", "what is in the image")
    DEPLOYMENT_QUESTION = mappings.get("DEPLOYMENT_QUESTION", FIXED_QUESTION)
    MODEL_IMAGE_SIZE = tuple(mappings.get("MODEL_IMAGE_SIZE", (224, 224)))

if "model" not in globals() and os.path.exists(MODEL_ARTIFACT_PATH):
    model = tf.keras.models.load_model(MODEL_ARTIFACT_PATH, compile=False)

MODEL_OBJECT = globals().get("model")
if MODEL_OBJECT is None:
    raise RuntimeError("Train or load the model before running inference helpers.")


def clean_image(image, target_size=None):
    if image is None:
        return None
    image = image.convert("RGB")
    image = ImageOps.autocontrast(image)
    image = image.filter(ImageFilter.MedianFilter(size=3))
    if target_size is not None:
        image = image.resize(target_size)
    return image


def encode_question_for_gui(question=None):
    question = question or DEPLOYMENT_QUESTION
    return encode_question(question)


def encode_image_for_model(image):
    cleaned = clean_image(image, MODEL_IMAGE_SIZE)
    image_array = np.array(cleaned, dtype=np.float32)
    image_array = convnext_preprocess_input(image_array)
    return np.expand_dims(image_array, axis=0)


def predict_model_answer(image, question=None):
    question = question or DEPLOYMENT_QUESTION
    image_array = encode_image_for_model(image)
    question_encoded = np.expand_dims(encode_question_for_gui(question), axis=0)
    preds = MODEL_OBJECT.predict({"image_input": image_array, "question_input": question_encoded}, verbose=0)

    pred_idx = int(np.argmax(preds[0]))
    pred_answer = idx2answer.get(pred_idx, "unknown")
    confidence = float(preds[0][pred_idx])
    top_indices = preds[0].argsort()[-5:][::-1]
    top_5 = [f"{idx2answer.get(int(idx), 'unknown')} : {float(preds[0][int(idx)]):.4f}" for idx in top_indices]
    return pred_answer, confidence, "\n".join(top_5)


def collapse_spaced_letters(text):
    text = "" if text is None else str(text).strip()
    parts = text.split()
    if len(parts) >= 2 and all(len(part) == 1 for part in parts):
        return "".join(parts)
    return text


def clean_spoken_label(text):
    text = collapse_spaced_letters(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "unclear"


ARABIC_LABELS = {
    "yes": "نعم",
    "no": "لا",
    "unclear": "غير واضحة",
    "unknown": "غير واضحة",
    "traffic sign": "علامة مرور",
    "traffic light": "إشارة مرور ضوئية",
    "traffic signal": "إشارة مرور",
    "street sign": "لافتة شارع",
    "sign": "علامة",
    "stop": "قف",
    "stop sign": "علامة قف",
    "person": "شخص",
    "car": "سيارة",
    "bus": "أتوبيس",
    "bottle": "زجاجة",
    "chair": "كرسي",
    "door": "باب",
    "phone": "هاتف",
    "cell phone": "هاتف محمول",
}


def answer_to_arabic(answer):
    label = clean_spoken_label(answer)
    key = label.lower()
    if key in ARABIC_LABELS:
        return ARABIC_LABELS[key]
    if re.search(r"[\u0600-\u06ff]", label):
        return label
    try:
        translated = GoogleTranslator(source="auto", target="ar").translate(label)
        return translated if translated else "غير واضحة"
    except Exception:
        return "غير واضحة"


def save_tts_audio(text, lang):
    text = str(text).strip() or ("غير واضحة" if lang == "ar" else "unclear")
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_file.close()
    gTTS(text=text, lang=lang, slow=False).save(temp_file.name)
    return temp_file.name


def build_spoken_sentences(answer):
    label = clean_spoken_label(answer)
    english_text = f"The answer is {label}."
    arabic_text = f"الإجابة هي {answer_to_arabic(label)}."
    return english_text, arabic_text


def audio_html(path, label):
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return HTML(f"<div style='margin:8px 0'><b>{label}</b><br><audio controls src='data:audio/mpeg;base64,{encoded}'></audio></div>")


In [ ]:
import ipywidgets as widgets

upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Upload Image")
question_widget = widgets.Text(value=DEPLOYMENT_QUESTION, description="Question:", layout=widgets.Layout(width="80%"))
run_button = widgets.Button(description="Generate Audio TTS", button_style="primary", icon="volume-up")
gui_output = widgets.Output()


def get_uploaded_image(upload):
    value = upload.value
    if not value:
        return None
    if isinstance(value, dict):
        first = next(iter(value.values()))
        content = first["content"]
    else:
        first = value[0]
        content = first["content"]
    return Image.open(io.BytesIO(content)).convert("RGB")


def on_generate_audio(_button):
    with gui_output:
        clear_output()
        image = get_uploaded_image(upload_widget)
        if image is None:
            display(HTML("<b>Please upload one image first.</b>"))
            return

        question = question_widget.value.strip() or DEPLOYMENT_QUESTION
        answer, confidence, top_5 = predict_model_answer(image, question)
        english_text, arabic_text = build_spoken_sentences(answer)
        english_audio = save_tts_audio(english_text, "en")
        arabic_audio = save_tts_audio(arabic_text, "ar")

        display(image.resize((260, 260)))
        display(HTML(f"<b>Question:</b> {question}<br><b>Prediction:</b> {answer}<br><b>Confidence:</b> {confidence:.4f}<br><b>Model:</b> {MODEL_NAME}"))
        display(HTML(f"<pre>{top_5}</pre>"))
        display(HTML(f"<b>English TTS:</b> {english_text}"))
        display(HTML(f"<b>Arabic TTS:</b> {arabic_text}"))
        display(audio_html(english_audio, "English Audio TTS"))
        display(audio_html(arabic_audio, "Arabic Audio TTS"))


run_button.on_click(on_generate_audio)
display(widgets.VBox([upload_widget, question_widget, run_button, gui_output]))


## 12. Download Artifacts

Run this cell in Colab after training if you want to download the model and mapping files.


In [ ]:
from google.colab import files

files.download(MODEL_ARTIFACT_PATH)
files.download(MAPPING_ARTIFACT_PATH)
